# **Breast Cancer Detection Example**
## Training on Real World Data

This notebook is for users who understand how to load in data and give zero2neuro arguments. In this notebook we will be using the breast cancer example to delve a little deeper into using a real world dataset in zero2neuro.

By the end of this notebook you'll know how to:

- Inspect a dataset using pandas
- Specify input features and the target variable
- Train a deep neural network on a real world dataset
- Create problem specific visualizations
- Interpret classification results and model performance

In [ ]:
import os
import sys

# Optional if you don't have the neuro path variable set up in your bashrc (Set to folder/directory above keras3_tools and zero2neuro)
# os.environ["NEURO_REPOSITORY_PATH"] = "/home/myuser/neuro"

neuro_path = os.getenv("NEURO_REPOSITORY_PATH")
assert neuro_path is not None, "Environment variable NEURO_REPOSITORY_PATH must be set to directory above zero2neuro and keras3_tools"

sys.path.append(neuro_path + '/zero2neuro/src/')

from zero2neuro import *
from parser import *

In [ ]:
parser = create_parser()

This dataset contains details of a potentially cancerous tumor and then the diagnosis of whether it is benign or malignant. The first task for this problem is to examine our dataset and grab our input and output features.

# Examining Dataset with Pandas
Like before we are going to set up our config files to build a model to try to solve this problem, first we need to look at the dataset to properly understand this problem. We will be using a python library called pandas to examine this dataset.

In [ ]:
import pandas as pd
# ../wdbc.csv is the default path of the dataset.
df = pd.read_csv("../wdbc.csv")

In [ ]:
# This command shows the first few rows of the dataset
df.head()

In [ ]:
# This gives you the size of the dataset, so we have 569 examples with 32 features
df.shape 

In [ ]:
# This gives up all the column names (our feature)
df.columns

In [ ]:
# This is a very useful tool to find missing data and to find the datatype for each feature. 
df.info()

# Grabbing Features
This dataset is cleaned up and ready for use, but in the real world with raw datasets there's a good chance that you will have to clean it up first before plugging it in. Preprocessing data is an important step and there's a lot of good resources out there to learn from.  

For Zero2Neuro we need to specify our input and output columns, but we can write some code to give us a list to copy and paste to avoid typos and make it quicker. One thing to note, the ID column contains an identifer for each example. This is not needed for our training so we can drop it, as long as you don't specify a column zero2neuro automatically ignores it.

In [ ]:
# It's common to call your input features x and output as y. Here we drop Diagnosis and ID
# for our inputs as diagnosis is our prediction and ID is irrelevant. 
x = df.drop(columns=["Diagnosis", "ID"], axis=1)
y = df["Diagnosis"]

In [ ]:
# This will give you the full list (of inputs) to throw into your data configuration
for name in x.columns:
    print(name)

# Data Configuration  
Go ahead and open up data_config.txt
```
--data_format=tabular
--data_file=../wdbc.csv
--data_set_type=fixed

# Desired output column contains one of two strings; these are
#  translated into the values 0 and 1, respectively
--data_columns_categorical_to_int
Diagnosis:B,M

# Input features
--data_inputs
# TODO: FILL IN

# Desired output
--data_outputs
Diagnosis

# Training batch size
# TODO: Choose a batch size for training.
# Consider how many samples should be processed before the model updates its weights.
--data_batch=???
```  
Go ahead and copy your input features and put them in as arguments for data_iputs (the line breaks count as seperate values). There's two new arguments here, firstly data_columns_categorical_to_int. Since our Diagnosis column is not numerical we have to translate it to something that a computer can understand, so we assign numerical values to B and M with the notation shown. 
  
Next we have data_batch, when training a network it takes in a certain amount of examples before it updates its internal parameters. A small batch size (8, 16, 32) gives noisy gradients and takes longer per epoch but can yield better generalization and introduces a bit of randomness that can be useful. A large batch size (128, 256, 512) gives a stable gradient and better utilizes system resources but requires more careful tuning of hyperparameters (parameters set outside of training) and has the risk of ending up in a local minima (a subpar performance). For this example I suggest choosing a smaller batch size, but feel free to experiment.  

# Experiment Configuration
Let's open up experiment_config.txt.
```
--experiment_name=breast_cancer

# Single output with sigmoid non-linearity calls for binary cross-entropy
--loss=binary_crossentropy
--metrics=binary_accuracy

# TODO: Choose a learning rate for training.
# Consider how quickly you want the model to update its weights.
--learning_rate=???

# TODO: Choose the maximum number of training epochs (how many times model goes through dataset while training).
# NOTE: Early stopping can stop training before this limit is reached.
--epochs=???

# Stop training when the monitored metric stops improving.
--early_stopping_monitor=loss

# TODO: Choose how many epochs to wait for improvement before stopping the training.
--early_stopping_patience=???

--results_path=./results
--output_file_base={args.experiment_name}_R{args.data_rotation:02d}
--save_model
--render_model

# Save the training set results
--log_training_set

# Report the results to a XLSX file
--report
--report_training
--report_training_ins
```

We'll go into more detail about it in the iris example but notice the loss and metric. For a binary classification we use binary crossentropy for our loss and binary accuracy for metrics. The reason, put simply, is that crossentropy is what our model trains off of and accuracy is meant to be a human readable metric for us to understand how the model is performing.

We've got learning rate also, to put it simply this is how big of a "step" the model takes when trying to find good performance when adjusting its parameters. A higher learning rate will make huge jumps but runs the risk of missing a global minima while too low of a learning rate can get stuck and also never find a global minima. Good starting values are 0.001-0.01.  
  
Early stopping monitors the model training performance and if the model hasn't improved in a certain amount of epochs (determined by patience) then the training will stop short. Your patience should be a certain fraction of your epochs, for this problem it's appropiate to pick a small number between 15-25.

# Network Configuration  
Once you finish up with the experiment configuration open up network_config.
```
--network_type=fully_connected
--input_shape
30

# TODO: Choose the number of neurons in each hidden layer.
# Try to balance the model's capacity for learning (ability to learn complex relationships) 
# with the simplicity of the network (generalized and lightweight).
--number_hidden_units
???
???

# TODO: Choose an activation function for the hidden layers.
--hidden_activation=???

# Single output encodes the probability of class M
--output_shape
1

# Use sigmoid with binary cross-entropy
--output_activation=sigmoid
```
We're doing another fully connected network, we have 30 input features, one output, it's binary so we use sigmoid. 

It's time to make your first proper network, experiment with the hidden units. You can add a hidden layer with each line break and the number you put is the number of neurons in that layer. The best place to start when constructing a network is to start small (first layer at around 16) and always funnel down, (for instance 128 -> 64 -> 32 -> 16 -> 8, but don't use this example in the network as this is way overkill for this problem).

There are many possibilites for how you can design your architecture, here are a few that you could try out.  
30 -> 16 -> 8 -> 1     # funneling  
30 -> 32 -> 16 -> 1    # expanding, then condensing  
30 -> 64 -> 32 -> 1    # wider than input  
30 -> 30 -> 16 -> 1    # same width as input, then condensing  
  
For the hidden activation it's good to use either elu or relu for this problem. 
  
Once you've filled out the config files, pass the files in and run the experiment.


In [ ]:
arg_string = "@network_config.txt @data_config.txt @experiment_config.txt -v --force"
args = parser.parse_args(arg_string.split())
print(args)

In [ ]:
prepare_and_execute_experiment(args)

# Examining Results
After successfully training your model, it's time to take a look at the results to see how well it did. You might also notice that in the training cell it has a binary_accuracy metric report, you can read this as an actual accuracy to how it did on the last epoch.

In [ ]:
# We open the pkl file like usual
with open('results/breast_cancer_R00_results.pkl', 'rb') as pickle_file:
    data = pickle.load(pickle_file) # Grab the data from pickle

In [ ]:
print(data.keys())
print(data['history'].keys())

So because the dataset has 569 examples, it's a lot to print out the raw predictions like we did for xor. If you want to examine the raw predictions it is provided in the .xlsx results file. Instead let's look at some visualizations.  

Firstly let's plot the epoch/loss, same as we did for xor.

In [ ]:
plt.plot(data['history']['loss'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Epoch/Loss')
plt.grid(True)
plt.show()

On top of loss we also have a metric we kept track of, our accuracy, so let's also plot that with respect to epochs.

In [ ]:
plt.plot(data['history']['binary_accuracy'])
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Epoch/Accuracy')
plt.grid(True)
plt.show()

Loss should be going down while accuracy should be going up.

## Confusion Matrix
This is a classification problem, which means there's a very specific visualization we can do to evaluate our model called a confusion matrix. This shows you true positives, false positives, true negatives, and false negatives. This is especially important in the medical context of this example, we do not ever want to be giving out false negatives when it comes to diagnosing someone. So let's use two new python libraries, sklearn and seaborn to create a confusion matrix.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

actual = data["outs_training"]

# Our model predicts a probability, to plot we need a flat integer so we set a threshold so that anything over 0.5
# is a 1 and anything below is a 0.
predicted = (data["predict_training"] >= 0.5).astype(int)

cm = confusion_matrix(actual, predicted)

# Here we set our xtick and ytick labels. Make sure you match these up with the categorical translations in data config.
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=["B", "M"], yticklabels=["B", "M"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()